In [65]:
from EncoderDecoder.utils import loadData
import matplotlib.pyplot as plt
import scipy.io
import logging
import pandas as pd
import numpy as np
import re
logging.disable(logging.CRITICAL)

In [2]:
import os

#add the root directory
os.chdir('../')

## Transform dataset received from Malik to correct format for training of encoder-decoder

In [4]:
path = "data-files/extraData/2022-Malik-LES-for-PCA/"
name = "species&prod_rates_mass_T_rho_fromCFLF_f05_forPCA.mat"

#### Load file

In [6]:
file_path = os.path.join(path, name)

matFile = scipy.io.loadmat(file_path)

print(matFile.keys())

dict_keys(['__header__', '__version__', '__globals__', 'T', 'prod_rates', 'rho', 'species_mass', 'species_names'])


#### Check basic information

In [15]:
print(f'Header: {matFile["__header__"]}')
print(f'Version: {matFile["__version__"]}')
print(f'Globals: {matFile["__globals__"]}')
print(f'Number of species: {len(matFile["species_names"])}')

Header: b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Mon Dec 23 21:30:32 2019'
Version: 1.0
Globals: []
Number of species: 35


#### Species names

In [27]:
print("Species names")
state_space_names_array = matFile["species_names"]
print(type(state_space_names_array))
print(state_space_names_array.shape)
state_space_names_array = [x[0] for x in state_space_names_array.flatten()]
print(state_space_names_array)

Species names
<class 'numpy.ndarray'>
(35, 1)
['H2', 'H', 'O', 'O2', 'OH', 'H2O', 'HO2', 'H2O2', 'C', 'CH', 'CH2', 'CH2S', 'CH3', 'CH4', 'CO', 'CO2', 'HCO', 'CH2O', 'CH2OH', 'CH3O', 'CH3OH', 'C2H', 'C2H2', 'C2H3', 'C2H4', 'C2H5', 'C2H6', 'HCCO', 'CH2CO', 'HCCOH', 'C3H7', 'C3H8', 'CH2CHO', 'CH3CHO', 'N2']


#### Create dataframes

In [28]:
data_state_space = pd.DataFrame(columns=state_space_names_array)
data_state_space_source = pd.DataFrame(columns=state_space_names_array)
data_mf = pd.DataFrame(columns=["f"])
data_T = pd.DataFrame(columns=["T"])

#### Temperature data

In [31]:
print("Temperature")
temp_array = matFile["T"]
print(type(temp_array))
print(temp_array.shape)
print(temp_array)
data_T = pd.DataFrame(temp_array, columns = ["T"])
print(data_T)

Temperature
<class 'numpy.ndarray'>
(89449, 1)
[[ 320.   ]
 [ 320.   ]
 [ 320.   ]
 ...
 [1355.001]
 [1355.001]
 [1355.   ]]
              T
0       320.000
1       320.000
2       320.000
3       320.000
4       320.000
...         ...
89444  1355.002
89445  1355.002
89446  1355.001
89447  1355.001
89448  1355.000

[89449 rows x 1 columns]


#### Species mass fractions data

In [32]:
print("Species mass fractions")
state_space_array = matFile["species_mass"]
print(type(state_space_array))
print(state_space_array.shape)
print(state_space_array)
data_state_space = pd.DataFrame(state_space_array, columns = state_space_names_array)
print(data_state_space)

Species mass fractions
<class 'numpy.ndarray'>
(89849, 35)
[[5.04269744e-09 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  6.94355593e-23 5.96100339e-01]
 [5.60134288e-09 2.55754737e-31 1.13224326e-49 ... 0.00000000e+00
  3.61595494e-23 5.96100339e-01]
 [6.22183280e-09 5.68830145e-35 1.56938953e-49 ... 0.00000000e+00
  1.12238154e-22 5.96100338e-01]
 ...
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 7.56170172e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 7.56573346e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 7.56976520e-01]]
                 H2             H             O        O2            OH  \
0      5.042697e-09  0.000000e+00  0.000000e+00  0.188558  0.000000e+00   
1      5.601343e-09  2.557547e-31  1.132243e-49  0.188558  7.285347e-26   
2      6.221833e-09  5.688301e-35  1.569390e-49  0.188558  8.242601e-30   
3      6.911121e-09  8.499935e-37  2.1819

#### Reaction rates/source terms data

In [33]:
print("Species reaction rates")
state_space_source_array = matFile["prod_rates"]
print(type(state_space_source_array))
print(state_space_source_array.shape)
print(state_space_source_array)
data_state_space_source = pd.DataFrame(state_space_source_array, columns = state_space_names_array)
print(data_state_space_source)

Species reaction rates
<class 'numpy.ndarray'>
(89849, 35)
[[-2.35687459e-27  1.47142194e-27  7.64245940e-46 ...  0.00000000e+00
  -1.98441055e-41  0.00000000e+00]
 [-1.24008376e-28 -1.04696718e-24  2.68201621e-32 ...  2.96911854e-46
  -1.25069820e-39  0.00000000e+00]
 [-5.20823218e-31 -3.23760887e-28  5.96512918e-36 ...  2.04976228e-49
  -3.25123179e-41  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]
                 H2             H             O            O2            OH  \
0     -2.356875e-27  1.471422e-27  7.642459e-46 -1.151117e-17  4.588383e-22   
1     -1.240084e-28 -1.046967e-24  2.682016e-32 -8.166908e-15 -6.430872e-21   
2     -5.208232e-31 -3.237609e-28  5.965129e-36 -2.584129e-19 -6.500

#### Create mixture fraction data

In [41]:
def get_mixture_fraction_from_equivalence_ratio(equivalence_ratio, Z_stoich):
    """
    This function computes mixture fraction vector based on the equivalence
    ratio and the stoichiometric mixture fraction using:

    equivalence_ratio = Z/(1 - Z) * (1 - Z_stoich)/Z_stoich

    Input:
    ----------
    `equivalence_ratio`
               - scalar or vector of equivalence ratio(s).
    `Z_stoich` - stoichiometric mixture fraction.

    Output:
    ----------
    `Z`        - vector of mixture fractions. Each element corresponds to the
                 element in `equivalence_ratio` vector.
    """

    if np.isscalar(equivalence_ratio):

        A = equivalence_ratio * Z_stoich / (1 - Z_stoich)
        Z = A / (1+A)

    elif len(equivalence_ratio) == 1:

        equivalence_ratio = np.asscalar(np.array(equivalence_ratio))

        A = equivalence_ratio * Z_stoich / (1 - Z_stoich)
        Z = A / (1+A)

    else:

        Z = np.empty([len(equivalence_ratio), 1])

        for i in range(0, len(equivalence_ratio)):

            A = equivalence_ratio[i] * Z_stoich / (1 - Z_stoich)
            Z[i] = A / (1+A)

    return Z

##### Check if first row corresponds to original value of oxidizer and fuel

In [46]:
max_oxygen = np.max(data_state_space["O2"])
oxygen_0 = data_state_space["O2"][0]
print(max_oxygen)
print(oxygen_0)

0.1885639005481575
0.18855787991127343


In [47]:
max_fuel = np.max(data_state_space["CH4"])
fuel_0 = data_state_space["CH4"][0]
print(max_fuel)
print(fuel_0)

0.21322169089334805
0.21322168981813555


In [ ]:
Y_oxidizer_in_air = data_state_space["O2"][0]/(data_state_space["O2"][0] + data_state_space["N2"][0])
print(Y_oxidizer_in_air)

# not correct since their max values are at different locations
Y_oxidizer_in_air = np.max(data_state_space["O2"])/(np.max(data_state_space["O2"]) + np.max(data_state_space["N2"]))
idx_O2 = np.argmax(data_state_space["O2"])
idx_N2 = np.argmax(data_state_space["N2"])

print(idx_O2, idx_N2)
print(Y_oxidizer_in_air)

0.24030574754485348
55527 67022
0.19942259939133247


Values correspond between first row and max value, but to be safe use max value as initial value

##### First method: compute mixture fraction via equivalence ratio

In [63]:
""" Equivalence ratio """

M_F = 16
M_O2 = 32
stoich_coeff = 2 # CH4 + 2*O2 -> CO2 + 2*H2O

Y_fuel_0 = 1
Y_oxidizer_0 = 0.232 #Y_oxidizer_in_air #reference 0.232

fuel_oxidizer_ratio = data_state_space["CH4"]/data_state_space["O2"]

fuel_oxidizer_ratio_stoich = M_F/(stoich_coeff*M_O2)

equivalence_ratio = fuel_oxidizer_ratio/fuel_oxidizer_ratio_stoich
print(equivalence_ratio)

mixture_fraction_stoich = 1/(1 + (Y_fuel_0 * M_O2 * stoich_coeff)/(Y_oxidizer_0 * M_F))
print(mixture_fraction_stoich)

mixture_fraction_method1 = get_mixture_fraction_from_equivalence_ratio(equivalence_ratio, mixture_fraction_stoich)
print(mixture_fraction_method1)

0        4.523209
1        4.523209
2        4.523209
3        4.523209
4        4.523209
           ...   
89844    0.060187
89845    0.045177
89846    0.030143
89847    0.015084
89848    0.000000
Length: 89849, dtype: float64
0.054820415879017016
[[0.20782425]
 [0.20782425]
 [0.20782425]
 ...
 [0.00174526]
 [0.00087412]
 [0.        ]]


##### Second method: compute mixture fraction directly

In [64]:
""" Mixture fraction """

s = 1/fuel_oxidizer_ratio_stoich #AFR_stoich

mixture_fraction = (s*data_state_space["CH4"] - data_state_space["O2"] + Y_oxidizer_0)/(s*Y_fuel_0 + Y_oxidizer_0)
print(mixture_fraction)

0        0.211798
1        0.211798
2        0.211798
3        0.211798
4        0.211798
           ...   
89844    0.023273
89845    0.022795
89846    0.022318
89847    0.021841
89848    0.021363
Length: 89849, dtype: float64


##### Third method with elements

In [78]:
# Atomic weights
atomic_weights = {
    "C": 12.011,
    "H": 1.008,
    "O": 15.999,
    "N": 14.007,
    "S": 32.06,
}

def parse_species_formula(species):
    """
    Convert species name into elemental composition.
    Example:
        CH4 -> {'C':1, 'H':4}
        CH3OH -> {'C':1, 'H':4, 'O':1}
        O2 -> {'O':2}
    """
    elements = re.findall(r"([A-Z][a-z]?)(\d*)", species)

    composition = {}
    for element, number in elements:
        composition[element] = int(number) if number else 1

    return composition


def compute_element_mass_fraction(data_state_space, species_names, element):
    """
    Compute elemental mass fraction Z_element from species mass fractions.
    """

    Z_element = np.zeros(len(data_state_space))

    for species in species_names:
        composition = parse_species_formula(species)

        if element not in composition:
            continue

        MW_species = sum(
            atomic_weights[e] * n
            for e, n in composition.items()
        )

        Z_element += (
            data_state_space[species]
            * composition[element]
            * atomic_weights[element]
            / MW_species
        )

    return Z_element


# Compute elemental mass fractions
Z_C = compute_element_mass_fraction(
    data_state_space,
    state_space_names_array,
    "C"
)

Z_H = compute_element_mass_fraction(
    data_state_space,
    state_space_names_array,
    "H"
)

Z_O = compute_element_mass_fraction(
    data_state_space,
    state_space_names_array,
    "O"
)


# Compute Bilger beta
beta = (
    2 * Z_C / atomic_weights["C"]
    + 0.5 * Z_H / atomic_weights["H"]
    - Z_O / atomic_weights["O"]
)

#print(beta)

beta_fuel = 4/(atomic_weights["C"] + 4 * atomic_weights["H"])
print(beta_fuel)

beta_oxidizer = -0.232/atomic_weights["O"]

mixture_fraction_method3 = (beta - beta_oxidizer)/(beta_fuel - beta_oxidizer)

print(mixture_fraction_method3)

print(type(mixture_fraction_method3))
data_mixture_fraction = pd.DataFrame(mixture_fraction_method3, columns = ["f"])
print(data_mixture_fraction)

0.24932992582434707
0        0.211794
1        0.211794
2        0.211794
3        0.211794
4        0.211794
           ...   
89844    0.023327
89845    0.022850
89846    0.022373
89847    0.021896
89848    0.021419
Length: 89849, dtype: float64
<class 'pandas.core.series.Series'>
              f
0      0.211794
1      0.211794
2      0.211794
3      0.211794
4      0.211794
...         ...
89844  0.023327
89845  0.022850
89846  0.022373
89847  0.021896
89848  0.021419

[89849 rows x 1 columns]


#### Save data into csv files

In [79]:
data_state_space.to_csv('data-files/Malik-state-space-CFLF-2022.csv', index=False)
data_state_space_source.to_csv('data-files/Malik-state-space_source-CFLF-2022.csv', index=False)
data_T.to_csv('data-files/Malik-T-CFLF-2022.csv', index=False)
data_mixture_fraction.to_csv('data-files/Malik-mf-CFLF-2022.csv', index=False)

In [80]:
np.savetxt("data-files/Malik-state-space-names-CFLF-2022.csv",(state_space_names_array), delimiter=',', fmt='%s')